In [3]:
import pandas as pd
import os
import glob

BASE_DIR = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Comps/RComps"
BASE_FILE = "KD_annotations_350.csv"
OUTPUT_DIR = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Comps/Results"

COLS = [
    "ASTrns", "ASUncc", "ASUnrg",
    "AktAcc", "AktAch", "AktAct", "AktSemel",
    "AktState",
    "ClCntfcl", "ClNegPol",
    "DtDTAM", "DtExp", "DtRes", "DtUniv",
    "TLAspPfv", "TLMdIrr", "TLTS",
    "VcPass",
]
KEY_COL = "TokenID"

os.makedirs(OUTPUT_DIR, exist_ok=True)

base = pd.read_csv(os.path.join(BASE_DIR, BASE_FILE)).set_index(KEY_COL)[COLS].sort_index()

compare_files = [
    f for f in glob.glob(os.path.join(BASE_DIR, "*.csv"))
    if os.path.basename(f) != BASE_FILE
]

print(f"Found {len(compare_files)} files to compare.\n")

rows = []

for comp_path in sorted(compare_files):
    comp_name = os.path.splitext(os.path.basename(comp_path))[0]
    comp = pd.read_csv(comp_path).set_index(KEY_COL)[COLS].sort_index()

    shared = base.index.intersection(comp.index)
    if len(shared) < len(base.index):
        print(f"  Note: {len(base.index) - len(shared)} base row(s) and {len(comp.index) - len(shared)} comp row(s) skipped (no matching TokenID)")
    base_aligned = base.loc[shared].fillna(0).astype(int)
    comp_aligned = comp.loc[shared].fillna(0).astype(int)

    mismatches = base_aligned != comp_aligned

    row = {"Run": comp_name}
    for c in COLS:
        row[c] = int(mismatches[c].sum())
        row[f"{c}_KD"] = int((mismatches[c] & (base_aligned[c] == 1)).sum())
    rows.append(row)
    print(f"Done: {comp_name}")

# Build summary DataFrame with interleaved columns (ASTrns, ASTrns_KD, ASUncc, ASUncc_KD, ...)
summary_cols = [col for c in COLS for col in (c, f"{c}_KD")]
summary = pd.DataFrame(rows).set_index("Run")[summary_cols]

out_path = os.path.join(OUTPUT_DIR, "mismatch_summary_r.csv")
summary.to_csv(out_path)
print(f"\nSummary saved to {out_path}")
summary

Found 2 files to compare.

  Note: 2 base row(s) and 2 comp row(s) skipped (no matching TokenID)
Done: Gemini_abl1
  Note: 2 base row(s) and 2 comp row(s) skipped (no matching TokenID)
Done: Gemini_full

Summary saved to /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/Project 1/Comps/Results/mismatch_summary_r.csv


,ASTrns,ASTrns_KD,ASUncc,ASUncc_KD,ASUnrg,ASUnrg_KD,AktAcc,AktAcc_KD,AktAch,AktAch_KD,...,DtUniv,DtUniv_KD,TLAspPfv,TLAspPfv_KD,TLMdIrr,TLMdIrr_KD,TLTS,TLTS_KD,VcPass,VcPass_KD
Run,,,,,,,,,,,,,,,,,,,,,
Gemini_abl1,62,52,51,48,20,12,90,63,138,116,...,49,30,109,104,12,5,58,1,7,2
Gemini_full,65,57,43,41,29,9,79,74,125,102,...,45,31,86,80,14,5,55,1,8,2
